# 从零开始构建推理引擎（一）：PyTorch 复现 Qwen3 推理过程

## 0. 准备虚拟环境

本文使用的 Python 版本为 3.11。读者可以使用自己青睐的虚拟环境工具，如 conda 或 virtualenv 或 uv 来创建本 notebook 所需的虚拟环境。

本文为读者提供了 `pyproject.toml` 描述虚拟环境所依赖的安装包，你可以使用 `uv sync` 或者其他方式来使用它。

## 1. 模型资源

### 1.1 下载模型资源

要实现某个模型的推理，首先需要获得模型的资源。
各个模型在开源的时候，一般都会在 huggingface 上发布自己的模型资源。
本文以 Qwen3-0.6B 为例，你可以在 [Qwen/Qwen3-0.6B](https://huggingface.co/Qwen/Qwen3-0.6B) 上找到模型资源。
模型资源一般使用 huggingface 提供的 [CLI](https://huggingface.co/docs/huggingface_hub/guides/cli) 来下载。

请首先按照 CLI 文档安装 `hf` 并下载 Qwen3-0.6B 的模型资源。
为了便于查看模型资源的内容，这次我们将模型资源下载到 `~/huggingface/Qwen3-0.6B/` 中。

In [ ]:
import os

os.environ["TOKENIZERS_PARALLELISM"] = "true"  # 取消 huggingface 检测到进程 fork 的警告

In [ ]:
!hf download Qwen/Qwen3-0.6B --local-dir ~/huggingface/Qwen3-0.6B/

### 1.2 模型资源介绍

我们需要了解一下模型资源的构成，明确推理过程会用到哪些资源文件。

#### 1.2.1 模型配置 `config.json`

模型配置文件 `config.json` 中记录了模型参数，比如词表大小、隐藏层维度、层数等。
发布的模型共享同一套代码，但可以有好几种变体和规格。
这些参数便是在描述模型规格的，用于在推理中初始化模型对象。

In [ ]:
!cat ~/huggingface/Qwen3-0.6B/config.json

#### 1.2.2 模型权重 `model.safetensors`

对于 Qwen3-0.6B 而言，所有层的权重都保存在 `model.safetensors` 文件中。
对于一些更大的模型，权重会保存在多个文件中。

模型权重文件可以通过 `safetensors` 库打开，我们可粗浅看一下这些权重的结构。
打开后的内容是一个字典，键为层名称，值为该层的权重张量。

In [ ]:
import os
from safetensors import safe_open

path = os.path.expanduser("~/huggingface/Qwen3-0.6B/model.safetensors")

tensors = {}

with safe_open(path, framework="pt", device="cpu") as f:
    for key in f.keys():
        tensors[key] = f.get_tensor(key)
        print(key, tensors[key].shape)

#### 1.2.3 分词器配置 `tokenizer.json`

模型使用的分词器配置，用于将输入文本转换为模型可处理的输入。
举个例子，分词器将文本 `hello\w` 转换为 `hel` 和 `lo\w` 两个 token，分别映射到数字 123 和 987。
推理引擎后续可以使用 `[123, 987]` 作为模型的输入。

Qwen3 模型使用的分词是 Byte-Pair Encoding（BPE），是一种基于统计的方法，
将文本中的单词或字符进行编码，并生成一个编码表，输出到 `tokenizer.json` 文件中。

另外还有一个 `tokenizer_config.json` 文件，用于配置分词器的参数，以及标记一些特殊 token。
根据 `tokenizer_config.json` 可以看到 transformers 库所使用的分词器类是 `Qwen2Tokenizer`。

实际上除了 transformers 库以外也存在多种分词库，有兴趣的读者还可以根据 BPE 算法和 `tokenizer.json` 自行实现一个分词器，并优化分词器的性能。
这里我们简单使用 transformers 库中的分词器即可，下面是使用分词器的一个简单例子。

In [ ]:
import os
from transformers import Qwen2Tokenizer

path = os.path.expanduser("~/huggingface/Qwen3-0.6B")

tokenizer = Qwen2Tokenizer.from_pretrained(path)
tokenizer.padding_side = "left"  # 推理需要左侧填充

# 用户原始的输入
prompts = ["Hello, how are you?", "Hello"]
# 通过 chat_template 函数填充用户的输入，具体可以见 tokenizer_config.json 中对应的脚本
chats = [[{"role": "user", "content": prompt}] for prompt in prompts]  # 两组多轮对话
texts = [
    tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True,
    )
    for messages in chats
]
print("-" * 40)
print(texts[0])
print("-" * 40)
# 对输入进行编码，生成 token id 张量，注意需要 padding
# 形状为 [batch_size, seq_len]
model_inputs = tokenizer([text for text in texts], return_tensors="pt", padding=True)
for k in model_inputs.keys():
    print(k)
    print(model_inputs[k])
    print(model_inputs[k].shape)
# 我们还可以对这些 token id 张量进行解码
print("-" * 40)
print(tokenizer.decode(model_inputs.input_ids[0]))
print("-" * 40)

另外旧版本的分词器所使用的是 `merges.txt` 和 `vocab.json`，模型资源中会给出但不一定会用到。
这两个文件所包含的内容与 `tokenizer.json` 是等价的。

#### 1.2.4 生成配置 `generation_config.json`

最后是生成配置，用于控制生成过程。
其中记录了生成过程中使用的参数，如温度、TopK、TopP 等。
在编写生成 token 的逻辑时，会用到这些内容。

生成配置文件 `generation_config.json` 的内容如下：

In [ ]:
!cat ~/huggingface/Qwen3-0.6B/generation_config.json

## 2. 使用 Transformers 进行推理

为了实现 Qwen3 的推理，除了准备好前面提到的模型资源以外，还需要准备模型推理代码。
模型在 huggingface 上发布的时候，已经提供了相应的推理代码，有助于我们理解模型结构并移植到任意的推理引擎中。

要复现 Qwen3 的推理，我们实际上需要理解 Qwen3 的模型结构。
首先我们从输入输出来只管感受一下 Qwen3 的推理。
我们可以复制并修改 huggingface 上 Qwen3 的官方示例并运行。

In [ ]:
import os
import random

import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# 设置种子以达到可复现的推理
seed = 42
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
random.seed(seed)

model_name = "~/huggingface/Qwen3-0.6B"

path = os.path.expanduser(model_name)

# 加载分词器和模型
tokenizer = AutoTokenizer.from_pretrained(path)
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    path,
    torch_dtype="auto",
    device_map="auto",
)

# 准备模型输入
prompts = [
    "Give me a short introduction to large language model.",
    "1 + 1 = ?",
]
chats = [[{"role": "user", "content": prompt}] for prompt in prompts]
texts = [
    tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True,
    )
    for messages in chats
]
model_inputs = tokenizer(texts, return_tensors="pt", padding=True).to(model.device)

# 将模型输入传进去，得到输出
generated_ids_list = model.generate(
    **model_inputs,
    max_new_tokens=256,
    do_sample=False,
)

for i, generated_ids in enumerate(generated_ids_list):
    # 去掉提示词的部分，之保留模型生成的内容
    output_ids = generated_ids[len(model_inputs.input_ids[i]) :].tolist()

    # 分离出 thinking 的部分
    try:
        # 从右开始找到第一个 151668 (</think>) 也就是特殊的 thinking 标记
        index = len(output_ids) - output_ids[::-1].index(151668)
    except ValueError:
        index = 0

    thinking_content = tokenizer.decode(
        output_ids[:index], skip_special_tokens=True
    ).strip("\n")
    content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

    print("-" * 40)
    print("thinking content:", thinking_content)
    print("content:", content)

## 3. 根据 Transformers 源码实现 Qwen3 模型

接下来我们根据 transformers 所实现的 Qwen3 模型来构建我们自己的 Qwen3 模型。
简单来说其实这里并不需要知道多少算法上的内容，但还是需要了解一下模型结构。

### 3.1 模型结构阅读

如果读者使用的是 uv 或者 virtualenv 管理的虚拟环境，可以打开
`.venv/lib/python3.11/site-packages/transformers/models/qwen3/modeling_qwen3.py` 来了解模型结构，
这份文件也可以在 transformers 的 github 仓库中找到。

这里是我的个人思考，我实现一个模型会从自底向上实现，主要是便于验证模型每一层、以及拼起来之后的正确性。
但在阅读模型结构的时候，则需要自顶向下阅读。下面是我阅读模型结构的一些心得。

#### 3.1.1 打印 model

其实有很多工具可以减轻阅读代码的负担。
首先 transformers 所初始化的 model 是可以打印的，我们直接来试一下

In [ ]:
import os

from transformers import AutoModelForCausalLM

model_name = "~/huggingface/Qwen3-0.6B"

path = os.path.expanduser(model_name)

model = AutoModelForCausalLM.from_pretrained(
    path,
    torch_dtype="auto",
    device_map="auto",
)

print(model)

这些输出可以与 `modeling_qwen3.py` 对照着看。
这里展示出了组合成入口类 `Qwen3ForCausalLM` 的所有和模型相关的成员变量。
通过这些成员变量，我们就可以得知模型有哪些权重。
不过很可惜的是这里并不会展示出一次推理，也就是 forward 过程，这些成员变量是如何调用和传递的。
而且除了 pytorch 自带的 `Module`，Qwen3 模型自己定义的 `Module` 由于缺少 `extra_repr` 函数的实现，因此无法展示输入和输出的形状。

#### 3.1.2 可视化工具

由于 pytorch 使用最为广泛，因此开源社区也为其提供了可视化工具。
这里选择 [torchinfo](https://github.com/TylerYep/torchinfo) 来介绍，当然还有其他的可视化方案，
如 [torchviz](https://github.com/szagoruyko/pytorchviz) 和 [netron](https://github.com/lutzroeder/netron) 等，
感兴趣的读者可以自行测试。

In [ ]:
import os

from transformers import AutoModelForCausalLM, AutoTokenizer
from torchinfo import summary

model_name = "~/huggingface/Qwen3-0.6B"

path = os.path.expanduser(model_name)

# 加载分词器和模型
tokenizer = AutoTokenizer.from_pretrained(path)
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    path,
    torch_dtype="auto",
    device_map="auto",
)

# 准备模型输入
prompts = [
    "Give me a short introduction to large language model.",
    "1 + 1 = ?",
]
chats = [[{"role": "user", "content": prompt}] for prompt in prompts]
texts = [
    tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True,
    )
    for messages in chats
]
model_inputs = tokenizer(texts, return_tensors="pt", padding=True).to(model.device)

# 我们按照 Qwen3ForCausalLM 中 forward 函数的输入，将 model_inputs 的各个成员传递进去
summary(
    model,
    input_data=[model_inputs.input_ids, model_inputs.attention_mask],
    col_names=["input_size", "output_size", "num_params"],
    depth=10,
    verbose=1,
)
pass  # 避免 summary 返回的 ModelStatistics 对象被重复打印

从输出中我们可以看到模型对组装好的输入是如何一步一步处理的。
我们后面的任务就是对着 `modeling_qwen3.py` 的源码，用 pytorch 抄一份 `Qwen3ForCausalLM` 出来。

### 3.2 实现模型的各个组件

这次我们先平铺实现一个模型，稍后我们再组装起来。

#### 3.2.1 嵌入 (Embedding)

Embedding 是 pytorch 已经提供的类，用来将每一个输入的索引 (token id) 映射到一个高维向量空间 (hidden states) 中。
举个例子，苹果的索引 (token id: 13) 被映射到向量 [0.9, 0.7, ...]，
其中第一个维度 0.9 表示“水果”，第二个维度 0.7 表示“好吃”。

这里我们准备了一个输入 `input_ids`，包含了两个请求，每个请求 8 个 token id。
Embedding 层将每个 token id 转换为一个 1024 维的向量，其拥有 151936 的词表大小。
运行下面的代码可以看到输入和输出的形状变化。

In [ ]:
import random

import numpy as np
import torch
from torch import nn

# 设置种子以达到可复现的推理
seed = 42
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
random.seed(seed)


padding_idx = 151645  # [PAD] 的 token id 值
vocab_size = 151936  # 词表大小，也就是词汇的种类
hidden_size = 1024  # 隐藏层维度

# [batch_size, seq_len]
input_ids = torch.arange(16).view(2, -1)
print(f"{input_ids.shape=}")

# [vocab_size, hidden_size]
embed = nn.Embedding(vocab_size, hidden_size, padding_idx)
print(f"{embed.weight.shape=}")

# [batch_size, seq_len, hidden_size]
hidden_states = embed(input_ids)
print(f"{hidden_states.shape=}")

#### 3.2.2 残差 (Residual)

接下来是迭代多次的 Decode Layer。
我们先只关注第一层，因为后面几层都是一样的结构。
首先是保留一下 hidden state，用于后续的残差连接。

In [ ]:
residual = hidden_states
# hidden_states = Function(hidden_states, ...)
# hidden_states = hidden_states + residual

#### 3.2.3 均方根归一化 (RMSNorm)

均方根归一化是将输入向量的每个元素除以它的均方根。
另外还需为每个元素添加缩放因子，并可配置全局偏移参数。
见下面的公式

$$
\begin{aligned}
\mathbf{y} & = \frac{\mathbf{x}}{\sqrt{\frac{\sum_i^n x_i^2}{n} + \epsilon}}   \odot \mathbf{w}
\end{aligned}
$$

接下来是我们的实现

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, hidden_size: int, eps=1e-6):
        super().__init__()
        # [hidden_size]
        self.weight = nn.Parameter(torch.ones(hidden_size))
        self.eps = eps

    def forward(self, hidden_states: torch.Tensor):
        # input: [batch_size, seq_len, hidden_size]
        # [batch_size, seq_len, 1]
        variance = hidden_states.pow(2).mean(-1, keepdim=True)
        # output: [batch_size, seq_len, hidden_size]
        return hidden_states * torch.rsqrt(variance + self.eps) * self.weight


rms_norm = RMSNorm(hidden_size)
print(f"{rms_norm.weight.shape=}")

hidden_states = rms_norm(hidden_states)
print(f"{hidden_states.shape=}")

#### 3.2.4 注意力 (Attention)

注意力机制可以说是模型中的重头戏。
Qwen3 中使用的是 GQA 注意力机制。

对于原始的 Attention 机制，输入向量会映射到三个空间中，
分别对应 Query、Key 和 Value。
这里提供一个不太精确的解释：
Query 空间描述的是输入向量的一些查询，其维度相当于提问的角度。
Key 空间描述的是输入向量的一些属性，其维度相当于回答的角度。
那么 Query 和 Key 空间的乘积就是查询和属性的关联度。
比如 *The quick brown fox jumps over the lazy dog* 这个例子，
`Q(quick) * K(fox)` 就要比 `Q(quick) * K(the)` 更高。
Value 空间描述的输入向量原生的受关注度。
比如 `V(jumps)` 就比 `V(the)` 更高。

我们假设输入向量的维度为 1024，Q、K、V 的维度为 2048，序列长度为 10，就有下面的计算流程。
为了方便起见省略了最外层的 batch 维度。

![](https://mermaid.ink/svg/pako:eNqVk11vmzAUhv-KdZRLGmEIgXGxm7ZX_diWRN20nTZygxMQASe2ydoS_vuwKVOktlHjK9vyc57XR3YNC5FwiGG5Fn8XKZOaXE-wJO14-oMwGPxCzErEgun08bGeNA81dRF1VnBFqOuNmsEA4Z6cnX0le3P-53z7DuGNesZzR1HLOORWyAJhT7ZW88NA5APPK9N6XpMd6vLTdLnVXR251VHb7jM249lZz90Jnm3nmYjvl3uiLD492vxDOO9DGtwhM8lKtRGKmyiqO6L6I9MFW3PSwriUbFHTpkZUW6nrZJ43tlk3TOWGZFqXc2GjIM5SrtkJ99lZX1ei2-nmh-0Ub-qZKm_e1550Ib596jGCAyuZJRBrWXEHCi4LZpZQmxgIOuVF25i4nSZ8yaq1RsCyabENK38LUfSkFNUqhXjJ1qpdVZuEaX6RsZVkxf9dycuEy3NRlRpiSn1bBOIaniAej4cR9YKxF9LQd70wdOAZYj_6MozGfhBQN4yiyA-DxoEXq3WHURi4B4M6wJNMC3nT_VL7WZt_VMUpaw)

多头注意力 (MHA) 机制是对原始的 Attention 机制的扩展，
将原始的向量空间拆分多个子空间，然后在 Q、K、V 各自的子空间中进行计算。
我们沿用上面的例子，将 Q、K、V 分成 8 个子空间。

![](https://mermaid.ink/svg/pako:eNqdlEuPmzAUhf-KdZVFK5EIAyGERTd9bNrpIzNqq9Zt5AQnIAImxjBpCf-9NiZVmmmjoayM5XPO53ttN7DmEYMQNjt-v46pkOjNguRIfYevBEajz4QkOSEZlfFq1Sza7w22CZFJxkqEbcdrRyMC39B4_Awd9fpPy_1fFI530ji2FyiNhd5ykRE4on0X80GL0D9yeo3KMWR7E1cWu0SiQFn0HsvkqsvUPzc5nDOnw5jTLu_1ldJcIqcXyGlvYZD_g7h-DLFmrbugjwNY6wvWurcYwlpWq62gRYwIvOICJUj1RdB8y57YVmDhpwT6Vva9XPD3L4-o7IJurwZh-4-6pqeiaAcL3amQsuAl01sve5bTkts13TGk1GQj6LrBbUNIuReyiZZp2zX4hpapVlIp8yXnHQ4hdzGTdMjua1PC3sXMsTwyg37WQGVMbJkuspk9DxzQMSM-Px78gVprHlzcIzKR7x51y8GCrUgiCKWomAWKPaP6FxqNQUDGLFOFD9UwYhta7aRuc6tkBc2_cJ6dlIJX2xjCDd2V6q8qIirZi4SqE5P9nhWqYEw851UuIXSdudu5QNjAAcIxDoKJa7szPJtNbey5Mwt-QOh5_mQ-dQPsBp6NXddtLfjZ5eLJ1ME-9gPsTx3bn2MLWJRILm7M89e9gu0vMQiYOQ)

GQA 机制是在 MHA 的基础上，让多个 Q 子空间查询同一个 K 子空间，并和同一个 V 子空间进行关联。
我们沿用上面的例子，将 Q 分成 8 个子空间，K 和 V 分成 4 个子空间。

![](https://mermaid.ink/svg/pako:eNqVlF-TkzAUxb9K5k4fdKQdApSyPPjinxddddsddTTKpCUtDIW0IWCV8t1NCHVr3a1dnsIdzjk_7r3QwILHDEJYrvmPRUKFRG-npEDq2n0lMBh8JiQtCMmpTObzZtp-b7BNiExzViJsO147GBD4hobD52ivn_8Ube9RON5B49heoDQWesdFTmCPtl3MjRahB3J6jcoxZFsTV27WqUSBsug9otRCN1GTPsPtWbuxf-y2O4bPzsL3L3wHn3XBby7qkUnLjtk95dBbROkZk3PE9SXEmrXugj4-grU-Ya17i8ewltV8JegmQQRec4FSpOYiaLFiT2wrsJynBPqZ9kOd8g-v9qjsgmZqoLO7eT5I_Vd_s0NztJOFblVYueEl0y0oe6bDI7MFXTOk1GQp6KLBbUNIuRWyiaOs7QZ9TctMK6mURcR5h0XIbcIkVXCH0_8QT3pSm8b2nqbGitgc-qpBzJlYMb3ipnocfy7u5Hsx4uOl4f-oteaepTGR7y9aGrBgJdIYQikqZoFiz6m-hUZjEJAJy9UYQnWM2ZJWa6mH3yrZhhZfOM8PSsGrVQLhkq5LdVdtYirZy5SqPcr_VIVqGBMveFVICF3bxZ0LhA3sIBxO7JGq-VfYdR0P-45nwU8Ix7Y3uhq7AZ74E8_3At9tLfjVBePR2ME-9gPsjx1bCS1gcSq5uDa_x-4v2f4GL3aikA)

注意到过程中有两个组件，一个是旋转位置编码 (RoPE)，另一个是注意力掩码 (Attention Mask)。
为了实现 Attention 层，我们还需要实现这两个的计算。

##### 3.2.4.1 旋转位置编码 (RoPE)

RoPE 作用在 Q 和 K 上，使得 Q 和 K 在位置上具有编码信息。
理论在此处不展开，主要看计算流程。

计算 RoPE 时，我们需要有输入，也就是 Q、K 中的向量 $\mathbf{x}$，
以及其在句子中的位置 $m$。
另外还需要 Q、K 的子空间维度 $d$，以及旋转因子参数 $base$。
我们先计算旋转因子：

$\theta_i = base^{-2i/d}$

然后对 $\mathbf{x}$ 中每两个元素做旋转：

$$
\left(
\begin{array}{cccccc}
\cos m\theta_0 & -\sin m\theta_0 & 0 & 0 & \cdots & 0 \\
\sin m\theta_0 & \cos m\theta_0  & 0 & 0 & \cdots & 0 \\
0 & 0 & \cos m\theta_1 & -\sin m\theta_1 & \cdots & 0 \\
0 & 0 & \sin m\theta_1 & \cos m\theta_1  & \cdots & 0 \\
\vdots & \vdots & \vdots & \vdots & \ddots & \vdots \\
0 & 0 & 0 & 0 & \cos m\theta_{d/2-1} & -\sin m\theta_{d/2-1} \\
0 & 0 & 0 & 0 & \sin m\theta_{d/2-1} & \cos m\theta_{d/2-1}
\end{array}
\right)
\begin{pmatrix}
x_0 \\ x_1 \\ x_2 \\ x_3 \\ \vdots \\ x_{d-2} \\ x_{d-1}
\end{pmatrix}
$$

等价于

$$
\left(
\begin{array}{c}
x_0 \\
x_1 \\
x_2 \\
x_3 \\
\vdots \\
x_{d-2} \\
x_{d-1}
\end{array}
\right)
\otimes
\left(
\begin{array}{c}
\cos m\theta_0 \\
\cos m\theta_0 \\
\cos m\theta_1 \\
\cos m\theta_1 \\
\vdots \\
\cos m\theta_{d/2-1} \\
\cos m\theta_{d/2-1}
\end{array}
\right)
+
\left(
\begin{array}{c}
 -x_1 \\
  x_0 \\
 -x_3 \\
  x_2 \\
 \vdots \\
 -x_{d-1} \\
  x_{d-2}
\end{array}
\right)
\otimes
\left(
\begin{array}{c}
\sin m\theta_0 \\
\sin m\theta_0 \\
\sin m\theta_1 \\
\sin m\theta_1 \\
\vdots \\
\sin m\theta_{d/2-1} \\
\sin m\theta_{d/2-1}
\end{array}
\right)
$$

在 Qwen3 的代码中，RoPE 会将输入向量上下平均切开，如下所示

$$
\left(
\begin{array}{c}
x_0 \\
\vdots \\
x_{d/2-1} \\
x_{d/2} \\
\vdots \\
x_{d-1}
\end{array}
\right)
\otimes
\left(
\begin{array}{c}
\cos m\theta_0 \\
\vdots \\
\cos m\theta_{d/2-1} \\
\cos m\theta_0 \\
\vdots \\
\cos m\theta_{d/2-1}
\end{array}
\right)
+
\left(
\begin{array}{c}
 -x_{d/2} \\
 \vdots \\
 -x_{d-1} \\
  x_0 \\
  \vdots \\
  x_{d/2-1}
\end{array}
\right)
\otimes
\left(
\begin{array}{c}
\sin m\theta_0 \\
\vdots \\
\sin m\theta_{d/2-1} \\
\sin m\theta_0 \\
\vdots \\
\sin m\theta_{d/2-1}
\end{array}
\right)
$$

最后对于序列而言，每次计算都要用到相同的 $\cos m\theta_{i} 和 \sin m\theta_{i}$，因此我们可以先缓存好这部分的计算结果，避免重复计算。
我们按最后这个公式实现如下

In [ ]:
class RotaryEmbedding(nn.Module):
    def __init__(
        self,
        head_dim: int,
        max_position: int = 65536,
        base: float = 10000,
    ):
        super().__init__()
        theta = 1.0 / (base ** (torch.arange(0, head_dim, 2) / head_dim))
        m = torch.arange(max_position)
        m_theta = torch.outer(m, theta)
        cos = m_theta.cos()  # [max_position, head_dim//2]
        sin = m_theta.sin()  # [max_position, head_dim//2]
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)

    def forward(self, query, key):
        # input: [batch_size, num_heads, seq_len, head_dim]
        return self.apply_rotary_emb(query), self.apply_rotary_emb(key)

    def apply_rotary_emb(self, x):
        # input: [batch_size, num_heads, seq_len, head_dim]
        seq_len = x.shape[-2]
        cos = self.cos[:seq_len]  # [seq_len, head_dim//2]
        sin = self.sin[:seq_len]  # [seq_len, head_dim//2]
        x1, x2 = x.chunk(2, dim=-1)  # [batch_size, num_heads, seq_len, head_dim//2]
        y1 = x1 * cos - x2 * sin  # [batch_size, num_heads, seq_len, head_dim//2]
        y2 = x2 * cos + x1 * sin  # [batch_size, num_heads, seq_len, head_dim//2]
        return torch.cat([y1, y2], dim=-1)  # [batch_size, num_heads, seq_len, head_dim]


test_batch_size = 2
test_seq_len = 8
test_num_heads = 16
test_head_dim = 128

test_x = torch.ones(test_batch_size, test_num_heads, test_seq_len, test_head_dim)
print(f"{test_x.shape=}")

rotary_emb = RotaryEmbedding(head_dim=test_head_dim, max_position=65536, base=10000)
print(f"{rotary_emb.cos.shape=}")

test_y = rotary_emb(test_x, test_x)
print(f"{test_y[0].shape=}")

##### 3.2.4.2 因果注意力掩码

对于 Q 和 K 的乘积而言，因果注意力掩码用于让当前的 token 不去关注未来的 token。
因此对于一个长度为 5 的序列而言，Q 和 K 的掩码如下

|  | ■ | ■ | ■ | ■ | ■ |
| --- | --- | --- | --- | --- | --- |
| ■ | ■ | ⬚ | ⬚ | ⬚ | ⬚ |
| ■ | ■ | ■ | ⬚ | ⬚ | ⬚ |
| ■ | ■ | ■ | ■ | ⬚ | ⬚ |
| ■ | ■ | ■ | ■ | ■ | ⬚ |
| ■ | ■ | ■ | ■ | ■ | ■ |

还记得使用 transformers 推理的时候，如果输入的 prompt 长度不一致，tokenizer 会对输入进行左侧填充。
其输出除了 token ids 之外，还有 attention mask。
如下所示，batch 中第二个输入的 attention mask 不全为 1。

```plaintext
attention_mask
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1]])
torch.Size([2, 14])
```

因此对于一个长度为 5 的序列而言，如果前两个 token 是 [PAD]，Q 和 K 的掩码如下

|  | ⬚ | ⬚ | ■ | ■ | ■ |
| --- | --- | --- | --- | --- | --- |
| ⬚ | ⬚ | ⬚ | ⬚ | ⬚ | ⬚ |
| ⬚ | ⬚ | ⬚ | ⬚ | ⬚ | ⬚ |
| ■ | ⬚ | ⬚ | ■ | ⬚ | ⬚ |
| ■ | ⬚ | ⬚ | ■ | ■ | ⬚ |
| ■ | ⬚ | ⬚ | ■ | ■ | ■ |

在 Attention 层中，我们对 $S = Q \cdot K^T$ 进行掩码，然后传递给 softmax。
softmax 的性质是，如果输入的元素是 `-inf`，那么 softmax 的输出结果就是 0。
因此我们的掩码矩阵中，⬚ 是 `-inf`, ■ 是 0。
不过为了多种浮点精度的兼容，我们使用 `torch.finfo(dtype).min` 来代替 `-inf`。
实现如下

In [ ]:
def make_causal_mask(attention_mask: torch.Tensor, dtype: torch.dtype = torch.float32):
    # input: [batch_size, seq_len]
    seq_len = attention_mask.shape[1]
    # [seq_len, seq_len]
    causal_mask = torch.tril(torch.ones(seq_len, seq_len))
    # [batch_size, seq_len, seq_len]
    pad_mask = attention_mask.unsqueeze(1).expand(-1, seq_len, -1)
    # output: [batch_size, seq_len, seq_len]
    return (1 - causal_mask * pad_mask) * torch.finfo(dtype).min


test_attention_mask = torch.tensor([[1, 1, 1, 1, 1], [0, 0, 1, 1, 1]])

make_causal_mask(test_attention_mask)

##### 3.2.4.3 QK 多头配对

由于 Qwen3 采用的是 QGA 的结构，因此对于每个 K、V 而言，需要个多个 Q 进行计算。
比如在 8 个头的 Q 和 4 个头的 K、V 中，每两个 Q 配对一个 K、V。
如下式

$
\begin{aligned}
Q & = \begin{pmatrix} q_0 & q_1 & q_2 & q_3 & \cdots & q_{n-2} & q_{n-1} \end{pmatrix} \\
K & = \begin{pmatrix} k_0 & k_0 & k_1 & k_1 & \cdots & k_{\frac{n}{2}-1} & k_{\frac{n}{2}-1} \end{pmatrix} \\
V & = \begin{pmatrix} v_0 & v_0 & v_1 & v_1 & \cdots & v_{\frac{n}{2}-1} & v_{\frac{n}{2}-1} \end{pmatrix} \\
\end{aligned}
$

因此我们需要让 K 和 V 在头的维度上，交错复制一份，代码如下：

In [ ]:
def repeat_kv(x: torch.Tensor, n_rep: int):
    # input: [batch_size, num_heads, seq_len, head_dim]
    # 1. unsqueeze -> [batch_size, num_heads, 1, seq_len, head_dim]
    # 2. expand -> [batch_size, num_heads, n_rep, seq_len, head_dim]
    shape = x.shape
    x = x[:, :, None, :, :].expand(-1, -1, n_rep, -1, -1)
    # output: [batch_size, num_heads * n_rep, seq_len, head_dim]
    return x.reshape(shape[0], shape[1] * n_rep, shape[2], shape[3])


x = torch.arange(1 * 3 * 2 * 4).reshape(1, 3, 2, 4)
print(f"{x=}")

y = repeat_kv(x, 2)
print(f"{y=}")

##### 3.2.4.4 注意力层实现

现在我们已经实现了注意力层的所有组件，我们只需要按照前面的计算流程组合起来即可。
最后不要忘记做残差连接。

In [ ]:
class Qwen3Attention(nn.Module):
    def __init__(
        self,
        hidden_size: int,
        num_heads: int,
        num_kv_heads: int,
        head_dim: int,
        rms_norm_eps: float = 1e-6,
    ):
        super().__init__()
        self.head_dim = head_dim
        self.num_kv_groups = num_heads // num_kv_heads
        self.q_proj = nn.Linear(hidden_size, num_heads * head_dim)
        self.k_proj = nn.Linear(hidden_size, num_kv_heads * head_dim)
        self.v_proj = nn.Linear(hidden_size, num_kv_heads * head_dim)
        self.o_proj = nn.Linear(num_heads * head_dim, hidden_size)
        self.q_norm = RMSNorm(head_dim, rms_norm_eps)
        self.k_norm = RMSNorm(head_dim, rms_norm_eps)
        self.rotary_emb = RotaryEmbedding(head_dim)
        self.scaling = self.head_dim**-0.5

    def forward(self, hidden_states: torch.Tensor, attention_mask: torch.Tensor):
        # input: [batch_size, seq_len, hidden_size]
        input_shape = hidden_states.shape[:-1]
        # 切成多头后的形状 [batch_size, seq_len, ?, head_dim]
        hidden_shape = (*input_shape, -1, self.head_dim)
        # 1. q_proj -> [batch_size, seq_len, num_heads * head_dim]
        # 2. view -> [batch_size, seq_len, num_heads, head_dim]
        # 3. norm -> keep
        # 4. transpose -> [batch_size, num_heads, seq_len, head_dim]
        q = self.q_norm(self.q_proj(hidden_states).view(hidden_shape)).transpose(1, 2)
        # 1. k_proj -> [batch_size, seq_len, num_kv_heads * head_dim]
        # 2. view -> [batch_size, seq_len, num_kv_heads, head_dim]
        # 3. norm -> keep
        # 4. transpose -> [batch_size, num_kv_heads, seq_len, head_dim]
        k = self.k_norm(self.k_proj(hidden_states).view(hidden_shape)).transpose(1, 2)
        v = self.v_proj(hidden_states).view(hidden_shape).transpose(1, 2)
        # apply rotary -> keep
        q, k = self.rotary_emb(q, k)
        # repeat kv -> [batch_size, num_heads, seq_len, head_dim]
        kk = repeat_kv(k, self.num_kv_groups)
        vv = repeat_kv(v, self.num_kv_groups)
        # S = QK^T / sqrt(d) -> [batch_size, num_heads, seq_len, seq_len]
        s = torch.matmul(q, kk.transpose(-2, -1)) * self.scaling
        # causal mask : [batch_size, seq_len, seq_len]
        causal_mask = make_causal_mask(attention_mask)
        # apply causal mask -> [batch_size, num_heads, seq_len, seq_len]
        s = s + causal_mask[:, None, :, :]
        # apply softmax -> [batch_size, num_heads, seq_len, seq_len]
        s = nn.functional.softmax(s, dim=-1)
        # O = SV -> [batch_size, num_heads, seq_len, head_dim]
        o = torch.matmul(s, vv)
        # 1. transpose -> [batch_size, seq_len, num_heads, head_dim]
        # 2. reshape -> [batch_size, seq_len, num_heads * head_dim]
        o = o.transpose(1, 2).reshape(*input_shape, -1).contiguous()
        # o_proj -> [batch_size, seq_len, hidden_size]
        o = self.o_proj(o)
        return o


num_heads = 16
num_kv_heads = 8
head_dim = 128

attention_mask = torch.ones(hidden_states.shape[:-1])  # [batch_size, seq_len]

self_attn = Qwen3Attention(
    hidden_size=hidden_size,
    num_heads=num_heads,
    num_kv_heads=num_kv_heads,
    head_dim=head_dim,
)
print(f"{self_attn=}")

hidden_states = self_attn(hidden_states, attention_mask)
print(f"{hidden_states.shape=}")

# 残差连接
hidden_states = hidden_states + residual

#### 3.2.5 多层感知器 (MLP)

多层感知器用于让模型实现抽象和记忆、拟合复杂函数。
主要过程是将输入向量投射到隐藏层空间，通过非线性带权重的激活函数后，再映射回输出空间。

我们假设输入向量的维度为 1024，序列长度为 10，隐藏层空间维度为 3072，就有下面的计算流程。
为了方便起见省略了最外层的 batch 维度。

![](https://mermaid.ink/svg/pako:eNqVkl1LwzAUhv9KOfRig2506UfaXHij6I0iDETRzJEtWVtYm5IlbNr1v9uPbTIUmbnKCe97njfJqWApuQACq7XcLlOmtHU_pYXVrN0bBdt-oTQrKM2ZTheLalq_VxOXUp3lYmNNXOTXtk1hZo1GV9a-1T_PK1PWv5iQf7R5LkadbW-ZsoM8_QE5qWeHWGeshGlxMa0Vd7y7f_BM2QH5YNA6KZW9rFUMh72k7fud6nZesaU-IHmv4GehudwWP0O34B8vu7dkl_jxom-gBTiQqIwD0coIB3KhctaWULVBKOhU5IICabZcrJhZawq0qBtbyYpXKfOjU0mTpEBWbL1pKlPy5o43GUsUy0-nShRcqGtpCg0kjOOuCZAKdkC8yRhFKA5ChHAQBmHkwAcQH6OxF_g4xrHrBS72awc-O6o7xhGO4iAIXRT6QYR9BwTPtFQP_YB2c1p_AY6U1Zs)

Qwen3 中使用的激活函数 $F_{act}$ 是 SiLU。
接下来是具体实现。
和进入 Attention 层一样，进入 MLP 层之前，先保留一下输入向量，然后对输入向量进行归一化。

In [ ]:
class Qwen3MLP(nn.Module):
    def __init__(
        self,
        hidden_size: int,
        intermediate_size: int,
        act_fn,
    ):
        super().__init__()
        self.hidden_size = hidden_size
        self.intermediate_size = intermediate_size
        self.act_fn = act_fn
        self.up_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.gate_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.down_proj = nn.Linear(intermediate_size, hidden_size, bias=False)

    def forward(self, hidden_states: torch.Tensor):
        # input: [batch_size, seq_len, hidden_size]
        # 1. gate_proj -> [batch_size, seq_len, intermediate_size]
        # 2. act_fn -> keep
        act_gate = self.act_fn(self.gate_proj(hidden_states))
        # up_proj -> [batch_size, seq_len, intermediate_size]
        up = self.up_proj(hidden_states)
        # down_proj -> [batch_size, seq_len, hidden_size]
        return self.down_proj(act_gate * up)


intermediate_size = 3072
act_fn = nn.functional.silu

mlp = Qwen3MLP(
    hidden_size=hidden_size, intermediate_size=intermediate_size, act_fn=act_fn
)

print(mlp)

residual = hidden_states
hidden_states = rms_norm(hidden_states)
hidden_states = mlp(hidden_states)
hidden_states = residual + hidden_states
print(f"{hidden_states.shape}")

#### 3.2.6 模型头 (LMHead)

根据前文所介绍的 Qwen3 模型结构，输入向量会经过多次 RMSNorm -> Attention -> RMSNorm -> MLP，
最后输出结果向量。
结果向量会作为模型头输入，输出下一个词在词表中的各个概率得分 (logits)。

在推理中，由于我们只需要预测下一个词，因此只需要计算每个序列中末尾 token 的 logits 即可。

In [ ]:
lm_head = nn.Linear(hidden_size, vocab_size, bias=False)
print(lm_head)

logits = lm_head(hidden_states[:, -1:, :])
print(f"{logits.shape=}")

#### 3.2.7 采样 (Sampling)

我们现在已经获得了每一个 batch 的下一个 token 在词表中的概率，
只需要从中间选择一个 token 就可以了。
常见的选择方法有不限于以下的多种方案：

- 随机选择：从概率分布中随机抽取一个 token。
- 贪心选择：从概率分布中选择概率最大的 token。
- nucleus 采样：从概率分布中选择概率最大的 token，直到概率和达到一个阈值。

这里我们简单起见用贪心来实现。

In [ ]:
sample_tokens = logits.argmax(dim=-1)
print(f"{sample_tokens}")

然后我们再和原来的输入拼起来，形成了一个新的句子。
这个句子比输入多一个词。

In [ ]:
output_ids = torch.cat([input_ids, sample_tokens], dim=1)
print(f"{output_ids}")

### 3.3 权重加载

前面我们实现了各个组件，按照模型资源文件中的 `config.json` 拼起来就好了。
但在此之前我们还需要了解一下如何将模型资源中的权重加载到我们写的模型当中去。

首先我们写一个示例嵌套的模型

In [ ]:
class ModuleB(nn.Module):
    def __init__(self):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(5, 10))  # 自定义参数
        self.weight.my_function = lambda: print("my_function")  # 自定义参数的自定义方法
        self.layers = nn.ModuleDict(
            {f"my_layer_{i}": nn.Linear(i, 1024, bias=False) for i in range(5)}
        )


class ModuleA(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_1 = nn.Linear(1024, 1024, bias=False)
        self.layers = nn.ModuleList([nn.Linear(i, 1024, bias=False) for i in range(5)])
        self.module_b = ModuleB()


module_a = ModuleA()
print(module_a)

下面我们展示如何找到对象 module_a 中的各个 tensor。

In [ ]:
for name, param in module_a.named_parameters():
    print(name, param.shape)

module_a.get_parameter("module_b.layers.my_layer_3.weight").shape

然后我们可以根据这样的寻址来将 safetensors 中的数据加载到模型当中去。
这里我们使用 Qwen3-0.6B 中的一个 1024 × 1024 的权重矩阵，加载到我们刚刚创建的示例模型当中去。

In [ ]:
import os
from safetensors import safe_open

path = os.path.expanduser("~/huggingface/Qwen3-0.6B/model.safetensors")

with safe_open(path, framework="pt", device="cpu") as f:
    key = "model.layers.0.self_attn.k_proj.weight"
    layer_tensor = f.get_tensor(key)
    print(key, layer_tensor.shape)


param = module_a.get_parameter("layer_1.weight")
param.data.copy_(layer_tensor)

在实际实现推理的模型部分时，为了最小化从 safetensors 加载到模型参数时的模型 key 时额外的转换逻辑，
我们最好按照 safetensors 中原本的模型层级结构来编写。

### 3.4 整合

接下来我们尝试将前面所有的内容整合在一起，形成一次完整的推理过程，文件见 `qwen3.py`。